In [ ]:

import wandb
from pathlib import Path
from dataclasses import dataclass, asdict


class WandbCallback:
    def __init__(self, log_every=50):
        self.log_every = log_every
        self.iteration = 0
        
    def __call__(self, env):
        # This gets called after each iteration
        if self.iteration % self.log_every == 0:
            # Log metrics to wandb
            metrics = {}
            for dataset_name, eval_name, value, _ in env.evaluation_result_list:
                metric_name = f"{dataset_name}/{eval_name}"
                metrics[metric_name] = value
            
            wandb.log(metrics, step=self.iteration)
        
        self.iteration += 1
        return False
    
@dataclass
class CFG:
    train_path: Path = Path("./data/train.csv")
    test_path: Path = Path("./data/test.csv")
    sub_path: Path = Path("./data/sample_submission.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.02

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    max_bin: int = 1024
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG()
asdict(cfg)

{'train_path': PosixPath('data/train.csv'),
 'test_path': PosixPath('data/test.csv'),
 'sub_path': PosixPath('data/sample_submission.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.01,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [27]:
from IPython.display import display
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess_df(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Replacing null values by median
    df['Episode_Length_minutes'].fillna(df['Episode_Length_minutes'].median(), inplace=True)
    df['Host_Popularity_percentage'].fillna(df['Host_Popularity_percentage'].median(), inplace=True)
    df['Guest_Popularity_percentage'].fillna(df['Guest_Popularity_percentage'].median(), inplace=True)
    df['Number_of_Ads'].fillna(int(df['Number_of_Ads'].median()), inplace=True)
    df['Episode_Num'].fillna(int(df['Episode_Num'].median()), inplace=True)

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    return df


df_train = pd.read_csv(cfg.train_path, index_col='id')
df_test = pd.read_csv(cfg.test_path, index_col='id')
df_sub = pd.read_csv(cfg.sub_path, index_col='id')

# is_dev_mode = False
# # is_dev_mode = True
# if is_dev_mode:
#     df_train = df_train.sample(10000, random_state=42)
#     df_test = df_test[:10]
#     df_sub = df_sub[:10]
    
df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

target_col = "Listening_Time_minutes"
y_train = df_train[target_col].copy()
df_train = df_train.drop(columns=[target_col])

display(df_train)
display(df_train.describe())

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Num
id,,,,,,,,,,
0,0,63.84,0,74.81,3,21,53.58,0.0,2,98
1,1,119.80,1,66.95,5,14,75.95,2.0,0,26
2,2,73.90,2,69.97,1,17,8.97,0.0,0,16
3,3,67.17,3,57.22,0,10,78.70,2.0,2,45
4,4,110.51,4,80.07,0,14,58.68,3.0,1,86
...,...,...,...,...,...,...,...,...,...,...
749995,36,75.66,2,69.36,5,10,53.58,0.0,0,25
749996,19,75.75,8,35.21,5,21,53.58,2.0,1,21
749997,37,30.98,9,78.58,3,10,84.89,0.0,0,51


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Num
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,23.540307,64.427546,4.556036,59.859901,3.030805,15.671500,52.498047,1.348854,0.997969,51.445811
std,13.917884,30.996996,2.965912,22.873098,2.024196,4.026379,25.537152,1.151130,0.815440,28.085623
min,0.000000,0.000000,0.000000,1.300000,0.000000,10.000000,0.000000,0.000000,0.000000,1.000000
25%,12.000000,39.420000,2.000000,39.410000,1.000000,14.000000,34.550000,0.000000,0.000000,28.000000
50%,23.000000,63.840000,5.000000,60.050000,3.000000,17.000000,53.580000,1.000000,1.000000,52.000000
75%,36.000000,90.310000,7.000000,79.530000,5.000000,21.000000,71.040000,2.000000,2.000000,75.000000
max,47.000000,325.240000,9.000000,119.460000,6.000000,21.000000,119.910000,103.910000,2.000000,100.000000


In [17]:
# Please get only duplicate with Podcast_Name, Episode_Num
df_dup = df_train[df_train.duplicated(subset=['Podcast_Name', 'Host_Popularity_percentage'], keep=False)]
df_dup = df_dup.sort_values(['Podcast_Name', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
id,,,,,,,,,,,
748552,Athlete's Arena,Episode 82,71.02,Sports,20.04,Friday,Afternoon,29.09,0.0,Neutral,56.09749
123261,Athlete's Arena,Episode 82,71.02,Music,20.04,Saturday,Afternoon,78.09,3.0,Positive,56.09749
707023,Athlete's Arena,Episode 82,71.02,Sports,20.04,Friday,Afternoon,78.09,0.0,Positive,56.09749
615974,Athlete's Arena,Episode 19,117.60,Sports,20.04,Sunday,Morning,NaN,2.0,Neutral,78.02501
88755,Athlete's Arena,Episode 63,72.54,Sports,20.06,Friday,Night,14.11,1.0,Negative,39.39000
...,...,...,...,...,...,...,...,...,...,...,...
162218,World Watch,Episode 71,44.50,News,99.89,Saturday,Evening,54.56,1.0,Negative,28.82105
381608,World Watch,Episode 57,16.63,News,99.89,Saturday,Evening,NaN,3.0,Negative,15.20660
491598,World Watch,Episode 50,74.06,News,99.89,Friday,Evening,NaN,2.0,Negative,50.76499


In [ ]:
# Please get only duplicate with Podcast_Name, Episode_Num
df_dup = df_train[df_train.duplicated(subset=['Podcast_Name', 'Episode_Num'], keep=False)]
df_dup = df_dup.sort_values(['Podcast_Name', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'])
df_dup

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Episode_Num
id,,,,,,,,,,
2221,0,55.10,0,68.79,6,14,6.29,1.0,2,1
6639,0,85.75,0,96.60,0,21,9.86,0.0,0,1
12478,0,69.75,0,94.31,4,17,94.08,2.0,0,1
26451,0,63.84,0,95.01,6,10,22.62,0.0,1,1
33972,0,90.33,0,85.02,0,10,27.76,3.0,2,1
...,...,...,...,...,...,...,...,...,...,...
714536,47,50.09,6,67.47,2,10,53.58,0.0,0,100
715096,47,71.71,6,73.30,1,21,53.58,1.0,1,100
717445,47,7.20,6,38.86,0,17,30.73,1.0,1,100


In [ ]:
!pip install python-dotenv


In [ ]:
int(len(df_train) * 0.2)

In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import gc
import wandb
import os
from dotenv import load_dotenv

load_dotenv()
wandb.login(key=os.getenv("WANDB_API_KEY"))
wandb.init(project="playground-series-s5e4", config=asdict(cfg))

X = df_train.copy()
y = y_train.copy()
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    test_size=0.2,  # 20% for validation
    random_state=42
)

X_test = df_test[X.columns].copy()

# # Target encoding if needed
# encoded_columns = df_train.columns[cfg.encoded_columns_start:]
# encoder = TargetEncoder(random_state=cfg.random_state)

# X_train[encoded_columns] = encoder.fit_transform(X_train[encoded_columns], y_train)
# X_valid[encoded_columns] = encoder.transform(X_valid[encoded_columns])
# X_test[encoded_columns] = encoder.transform(X_test[encoded_columns])

# Initialize the model
model = lgb.LGBMRegressor(
    n_iter=cfg.n_iter,
    max_depth=cfg.max_depth,
    num_leaves=cfg.num_leaves,
    colsample_bytree=cfg.colsample_bytree,
    learning_rate=cfg.learning_rate,
    objective=cfg.objective,
    metric=cfg.metric, 
    verbosity=cfg.verbosity,
    max_bin=cfg.max_bin,
)

# Train model with validation
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    callbacks=[
        lgb.log_evaluation(cfg.log_eval), 
        lgb.early_stopping(cfg.early_stopping),
        WandbCallback(log_every=10)
    ],
)

# Calculate validation score
val_score = model.best_score_['valid_1'][cfg.metric]
print(f"Validation score: {val_score}")

# Generate test predictions
y_pred = model.predict(X_test)

gc.collect()

Training until validation scores don't improve for 200 rounds
[100]	training's rmse: 17.2112	valid_1's rmse: 17.4211
[200]	training's rmse: 13.5709	valid_1's rmse: 14.04
[300]	training's rmse: 12.4985	valid_1's rmse: 13.1927
[400]	training's rmse: 12.0897	valid_1's rmse: 12.9637
[500]	training's rmse: 11.8044	valid_1's rmse: 12.855
[600]	training's rmse: 11.6215	valid_1's rmse: 12.8125
[700]	training's rmse: 11.4522	valid_1's rmse: 12.7851
[800]	training's rmse: 11.3007	valid_1's rmse: 12.7651
[900]	training's rmse: 11.1678	valid_1's rmse: 12.7499
[1000]	training's rmse: 11.0478	valid_1's rmse: 12.7373
[1100]	training's rmse: 10.9412	valid_1's rmse: 12.7267
[1200]	training's rmse: 10.8334	valid_1's rmse: 12.7172
[1300]	training's rmse: 10.7324	valid_1's rmse: 12.7086
[1400]	training's rmse: 10.637	valid_1's rmse: 12.7017
[1500]	training's rmse: 10.5476	valid_1's rmse: 12.6952
[1600]	training's rmse: 10.4581	valid_1's rmse: 12.6878
[1700]	training's rmse: 10.3686	valid_1's rmse: 12.6803